In [1]:
pip install duckdb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 13.2 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [3]:
import duckdb
print(duckdb.__version__)

1.5.6


In [4]:
con = duckdb.connect("/home/jovyan/work/analytics.duckdb")
print(con.execute("SELECT version()").fetchone())

('v1.5.6',)


In [5]:
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

In [4]:
con.execute("""
SET s3_endpoint='minio:9000';
SET s3_access_key_id='minio';
SET s3_secret_access_key='minio123';
SET s3_use_ssl=false;
SET s3_url_style='path';
""")

In [5]:
tables = {
    "dim_users": "s3a://end-to-end-streaming-data-platform-gold/dim_users",
    "dim_videos": "s3a://end-to-end-streaming-data-platform-gold/dim_videos",
    "fact_subscription": "s3a://end-to-end-streaming-data-platform-gold/fact_subscription",
    "fact_payments": "s3a://end-to-end-streaming-data-platform-gold/fact_payments",
    "fact_events": "s3a://end-to-end-streaming-data-platform-gold/fact_events",
}

for table_name, parquet_path in tables.items():
    con.execute(f"""
        CREATE OR REPLACE TABLE {table_name} AS
        SELECT *
        FROM read_parquet('{parquet_path}/*.parquet')
    """)

    print(f"Loaded {table_name}")

Loaded dim_users
Loaded dim_videos
Loaded fact_subscription
Loaded fact_payments
Loaded fact_events


In [6]:
tables = con.execute("SHOW TABLES").fetchall()

for table in tables:
    print(table[0])

dim_users
dim_videos
fact_events
fact_payments
fact_subscription


In [7]:
print(con.execute("SELECT current_database()").fetchone())
print(con.execute("SHOW TABLES").fetchall())

('analytics',)
[('dim_users',), ('dim_videos',), ('fact_events',), ('fact_payments',), ('fact_subscription',)]


In [8]:
print(con.execute("PRAGMA database_list").fetchall())

[(20655, 'analytics', '/home/jovyan/work/analytics.duckdb')]


In [9]:
print(con.execute("SHOW TABLES").fetchall())

[('dim_users',), ('dim_videos',), ('fact_events',), ('fact_payments',), ('fact_subscription',)]


In [6]:
for table_name, in con.execute("SHOW TABLES").fetchall():
    print(f"\n--- {table_name} ---")
    print(con.execute(
        f"DESCRIBE {table_name}"
    ).fetchall())


--- dim_users ---
[('user_id', 'VARCHAR', 'YES', None, None, None), ('email', 'VARCHAR', 'YES', None, None, None), ('country', 'VARCHAR', 'YES', None, None, None), ('account_status', 'VARCHAR', 'YES', None, None, None), ('created_at', 'DATE', 'YES', None, None, None), ('last_login', 'DATE', 'YES', None, None, None)]

--- dim_videos ---
[('video_id', 'VARCHAR', 'YES', None, None, None), ('match_id', 'VARCHAR', 'YES', None, None, None), ('content_type', 'VARCHAR', 'YES', None, None, None), ('status', 'VARCHAR', 'YES', None, None, None), ('title', 'VARCHAR', 'YES', None, None, None), ('duration_sec', 'INTEGER', 'YES', None, None, None), ('competition_name', 'VARCHAR', 'YES', None, None, None), ('technical_max_resolution', 'VARCHAR', 'YES', None, None, None), ('stats_views', 'INTEGER', 'YES', None, None, None), ('stats_likes', 'INTEGER', 'YES', None, None, None), ('language_1', 'VARCHAR', 'YES', None, None, None), ('language_2', 'VARCHAR', 'YES', None, None, None), ('created_at', 'DATE', 

In [6]:
tables = {
    "dim_users": "user_id",
    "dim_videos": "video_id",
    "fact_events": "event_id",
    "fact_payments": "payment_id",
    "fact_subscription": "subscription_id",
}

for table, key in tables.items():
    result = con.execute(f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(*) FILTER (
                WHERE {key} IS NULL
            ) AS null_keys,
            COUNT(DISTINCT {key}) AS unique_keys
        FROM {table}
    """).fetchone()

    print(
        f"{table}: "
        f"rows={result[0]}, "
        f"null_keys={result[1]}, "
        f"unique_keys={result[2]}"
    )

dim_users: rows=9997, null_keys=0, unique_keys=9997
dim_videos: rows=6235, null_keys=0, unique_keys=6235
fact_events: rows=8796, null_keys=0, unique_keys=8796
fact_payments: rows=11767, null_keys=0, unique_keys=11767
fact_subscription: rows=20140, null_keys=0, unique_keys=20140


In [9]:
query = """
SELECT
    'events -> users' AS relationship,
    COUNT(*) AS unmatched_rows
FROM fact_events e
LEFT JOIN dim_users u ON e.user_id = u.user_id
WHERE u.user_id IS NULL

UNION ALL

SELECT
    'events -> videos',
    COUNT(*)
FROM fact_events e
LEFT JOIN dim_videos v ON e.video_id = v.video_id
WHERE v.video_id IS NULL

UNION ALL

SELECT
    'subscriptions -> users',
    COUNT(*)
FROM fact_subscription s
LEFT JOIN dim_users u ON s.user_id = u.user_id
WHERE u.user_id IS NULL

UNION ALL

SELECT
    'payments -> subscriptions',
    COUNT(*)
FROM fact_payments p
LEFT JOIN fact_subscription s
    ON p.subscription_id = s.subscription_id
WHERE s.subscription_id IS NULL
"""

print(con.execute(query).fetchall())

[('events -> users', 5), ('events -> videos', 4763), ('subscriptions -> users', 6), ('payments -> subscriptions', 1)]


In [9]:
query = """
CREATE OR REPLACE VIEW vw_user_subscriptions AS
SELECT
    u.user_id,
    u.country,
    u.account_status,
    u.created_at AS user_created_at,
    u.last_login,
    s.subscription_id,
    s.plan_type,
    s.is_active,
    s.auto_renew,
    s.start_date,
    s.end_date
FROM dim_users AS u
INNER JOIN fact_subscription AS s
    ON u.user_id = s.user_id
"""

con.execute(query)

In [10]:
con.sql("""
    SELECT *
    FROM vw_user_subscriptions
    LIMIT 10
""").show()

┌──────────────────────────────────────┬─────────┬────────────────┬─────────────────┬────────────┬──────────────────────────────────────┬───────────┬───────────┬────────────┬────────────┬────────────┐
│               user_id                │ country │ account_status │ user_created_at │ last_login │           subscription_id            │ plan_type │ is_active │ auto_renew │ start_date │  end_date  │
│               varchar                │ varchar │    varchar     │      date       │    date    │               varchar                │  varchar  │  boolean  │  boolean   │    date    │    date    │
├──────────────────────────────────────┼─────────┼────────────────┼─────────────────┼────────────┼──────────────────────────────────────┼───────────┼───────────┼────────────┼────────────┼────────────┤
│ 01a0f05b-06bb-70ce-b48f-ad400d4c8b35 │ KWT     │ active         │ 2026-01-01      │ NULL       │ 01a0f05b-54dc-71ea-a21e-60911115844f │ free      │ true      │ false      │ 2026-01-01 │ 2026-01-

In [11]:
con.sql("""
SELECT
    (SELECT COUNT(*) FROM vw_user_subscriptions) AS view_rows,
    (SELECT COUNT(*)
     FROM fact_subscription s
     INNER JOIN dim_users u
         ON s.user_id = u.user_id) AS expected_rows,
    (SELECT COUNT(DISTINCT subscription_id)
     FROM vw_user_subscriptions) AS unique_subscriptions
""").show()

┌───────────┬───────────────┬──────────────────────┐
│ view_rows │ expected_rows │ unique_subscriptions │
│   int64   │     int64     │        int64         │
├───────────┼───────────────┼──────────────────────┤
│     20134 │         20134 │                20134 │
└───────────┴───────────────┴──────────────────────┘



In [5]:
query = """
CREATE OR REPLACE VIEW vw_subscription_payments AS
SELECT
    s.subscription_id,
    s.user_id,
    s.plan_type,
    s.is_active,
    s.start_date,
    s.end_date,
    p.payment_id,
    p.amount,
    p.currency,
    p.payment_status,
    p.payment_date
FROM fact_subscription AS s
INNER JOIN fact_payments AS p
    ON s.subscription_id = p.subscription_id
"""

con.execute(query)

In [6]:
con.sql("""
SELECT *
FROM vw_subscription_payments
LIMIT 10
""").show()

┌──────────────────────────────────────┬──────────────────────────────────────┬───────────┬───────────┬────────────┬────────────┬──────────────────────────────────────┬───────────────┬──────────┬────────────────┬──────────────┐
│           subscription_id            │               user_id                │ plan_type │ is_active │ start_date │  end_date  │              payment_id              │    amount     │ currency │ payment_status │ payment_date │
│               varchar                │               varchar                │  varchar  │  boolean  │    date    │    date    │               varchar                │ decimal(10,2) │ varchar  │    varchar     │     date     │
├──────────────────────────────────────┼──────────────────────────────────────┼───────────┼───────────┼────────────┼────────────┼──────────────────────────────────────┼───────────────┼──────────┼────────────────┼──────────────┤
│ 01a0f05b-54dd-7851-b25c-ba3fe8e290e0 │ 01a0f05b-06bb-70ce-b48f-ad400d4c8b35 │ premium 

In [7]:
con.sql("""
SELECT
    (SELECT COUNT(*)
     FROM vw_subscription_payments) AS view_rows,

    (SELECT COUNT(*)
     FROM fact_payments p
     INNER JOIN fact_subscription s
         ON p.subscription_id = s.subscription_id) AS expected_rows,

    (SELECT COUNT(DISTINCT payment_id)
     FROM vw_subscription_payments) AS unique_payments
""").show()

┌───────────┬───────────────┬─────────────────┐
│ view_rows │ expected_rows │ unique_payments │
│   int64   │     int64     │      int64      │
├───────────┼───────────────┼─────────────────┤
│     11766 │         11766 │           11766 │
└───────────┴───────────────┴─────────────────┘



In [8]:

query = """
CREATE OR REPLACE VIEW vw_video_engagement AS
SELECT
    e.event_id,
    e.event_timestamp,
    e.user_id,
    e.video_id,
    e.interaction_type,
    e.watch_time_sec,
    e.device_type,
    v.content_type,
    v.status AS video_status,
    v.title AS video_title,
    v.competition_name,
    v.technical_max_resolution
FROM fact_events AS e
INNER JOIN dim_videos AS v
    ON e.video_id = v.video_id
"""

con.execute(query)


In [9]:

con.sql("""
SELECT *
FROM vw_video_engagement
LIMIT 10
""").show()


┌──────────────────────────────────────┬─────────────────────┬──────────────────────────────────────┬──────────────────────────────────────┬──────────────────┬────────────────┬─────────────┬──────────────┬──────────────┬─────────────────────────┬──────────────────┬──────────────────────────┐
│               event_id               │   event_timestamp   │               user_id                │               video_id               │ interaction_type │ watch_time_sec │ device_type │ content_type │ video_status │       video_title       │ competition_name │ technical_max_resolution │
│               varchar                │      timestamp      │               varchar                │               varchar                │     varchar      │     int32      │   varchar   │   varchar    │   varchar    │         varchar         │     varchar      │         varchar          │
├──────────────────────────────────────┼─────────────────────┼──────────────────────────────────────┼────────────────────

In [10]:

con.sql("""
SELECT
    (SELECT COUNT(*)
     FROM vw_video_engagement) AS view_rows,

    (SELECT COUNT(*)
     FROM fact_events e
     INNER JOIN dim_videos v
         ON e.video_id = v.video_id) AS expected_rows,

    (SELECT COUNT(DISTINCT event_id)
     FROM vw_video_engagement) AS unique_events
""").show()


┌───────────┬───────────────┬───────────────┐
│ view_rows │ expected_rows │ unique_events │
│   int64   │     int64     │     int64     │
├───────────┼───────────────┼───────────────┤
│      4033 │          4033 │          4033 │
└───────────┴───────────────┴───────────────┘



In [11]:

query = """
CREATE OR REPLACE VIEW vw_user_engagement AS
SELECT
    e.event_id,
    e.event_timestamp,
    e.user_id,
    e.video_id,
    e.interaction_type,
    e.watch_time_sec,
    e.device_type,
    u.country,
    u.account_status,
    u.created_at AS user_created_at
FROM fact_events AS e
INNER JOIN dim_users AS u
    ON e.user_id = u.user_id
"""

con.execute(query)


In [12]:

con.sql("""
SELECT *
FROM vw_user_engagement
LIMIT 10
""").show()


┌──────────────────────────────────────┬─────────────────────┬──────────────────────────────────────┬──────────────────────────────────────┬──────────────────┬────────────────┬─────────────┬─────────┬────────────────┬─────────────────┐
│               event_id               │   event_timestamp   │               user_id                │               video_id               │ interaction_type │ watch_time_sec │ device_type │ country │ account_status │ user_created_at │
│               varchar                │      timestamp      │               varchar                │               varchar                │     varchar      │     int32      │   varchar   │ varchar │    varchar     │      date       │
├──────────────────────────────────────┼─────────────────────┼──────────────────────────────────────┼──────────────────────────────────────┼──────────────────┼────────────────┼─────────────┼─────────┼────────────────┼─────────────────┤
│ 01a0f05b-d18d-7129-84ba-f8e558db1014 │ 2026-01-01 07:4

In [13]:

con.sql("""
SELECT
    (SELECT COUNT(*)
     FROM vw_user_engagement) AS view_rows,

    (SELECT COUNT(*)
     FROM fact_events e
     INNER JOIN dim_users u
         ON e.user_id = u.user_id) AS expected_rows,

    (SELECT COUNT(DISTINCT event_id)
     FROM vw_user_engagement) AS unique_events
""").show()


┌───────────┬───────────────┬───────────────┐
│ view_rows │ expected_rows │ unique_events │
│   int64   │     int64     │     int64     │
├───────────┼───────────────┼───────────────┤
│      8791 │          8791 │          8791 │
└───────────┴───────────────┴───────────────┘



In [14]:
con.close()